# 02 — Data Cleaning & Region Mapping

## Objective

Turn the raw World Bank pull into a clean, documented, analysis-ready Country x Year panel, and build a transparent regional classification (`REGION_MAPPING.csv`).

## Region Mapping Methodology

The brief's 8-region framework (North America, Europe, Latin America, Middle East, Africa, East Asia, South & Southeast Asia, Oceania) is not an official statistical classification — no single source publishes exactly this split. Rather than inventing an ad-hoc, undocumented per-country assignment, this project builds the mapping from the **UN M49 standard** (via the widely-used ISO-3166-Countries-with-Regional-Codes reference table, sourced from the UN Statistics Division), applying a transparent sub-region → Project-007-region rule, with three documented manual overrides where UN M49's geographic classification diverges from standard market-research convention (e.g. Iran is UN M49 'Southern Asia' but is grouped Middle East here, matching how virtually every commercial market-research firm classifies it).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src/visualisation')
regions = pd.read_csv('../data/processed/REGION_MAPPING.csv')
print(regions.shape)
regions['Project_007_Region'].value_counts()

(217, 7)


Project_007_Region
Africa                    54
Europe                    53
Latin America             42
Middle East               19
Oceania                   19
South & Southeast Asia    19
East Asia                  7
North America              4
Name: count, dtype: int64

### The three documented overrides (see `Notes` column for full reasoning on any row)

In [2]:
regions[regions['Notes'] != ''][['Country','ISO3','Project_007_Region','Sub_Region','Notes']]

,Country,ISO3,Project_007_Region,Sub_Region,Notes
0,Algeria,DZA,Africa,Northern Africa,NaN
1,Angola,AGO,Africa,Sub-Saharan Africa,NaN
2,Benin,BEN,Africa,Sub-Saharan Africa,NaN
3,Botswana,BWA,Africa,Sub-Saharan Africa,NaN
4,Burkina Faso,BFA,Africa,Sub-Saharan Africa,NaN
...,...,...,...,...,...
212,Singapore,SGP,South & Southeast Asia,South-eastern Asia,NaN
213,Sri Lanka,LKA,South & Southeast Asia,Southern Asia,NaN
214,Thailand,THA,South & Southeast Asia,South-eastern Asia,NaN
215,Timor-Leste,TLS,South & Southeast Asia,South-eastern Asia,NaN


## Data Completeness Check

Before building the master panel, check completeness of every indicator for the most recent year — this determines whether 'most recent year' or 'latest available year per country' is the right approach for cross-sectional comparison (spoiler: the latter, since a fixed-year cut would silently exclude countries that report with a lag).

In [3]:
master_raw = pd.read_csv('../data/raw/worldbank_panel_wide.csv')
master_raw = master_raw[master_raw['iso3'].isin(regions['ISO3'])]
latest_fixed = master_raw[master_raw['year'] == 2023]
completeness = latest_fixed.drop(columns=['iso3','year','country_name']).notna().mean().sort_values()
completeness

household_consumption_expenditure_per_capita_constant_2015_usd    0.751152
household_consumption_expenditure_constant_2015_usd               0.751152
household_consumption_expenditure_current_usd                     0.788018
household_consumption_pct_of_gdp                                  0.792627
inflation_cpi_annual_pct                                          0.811060
internet_users_pct_of_population                                  0.834101
unemployment_pct_of_labor_force                                   0.847926
gdp_per_capita_ppp_current_intl                                   0.907834
gdp_current_usd                                                   0.940092
gdp_per_capita_current_usd                                        0.940092
population_total                                                  1.000000
urban_population_pct                                              1.000000
population_65_plus_pct                                            1.000000
age_dependency_ratio_pct 

**Decision: use "latest available year per country" (2013–2023 window), not a fixed year.** A fixed-year cut (2023 only) would have dropped real countries that happen to report with a one- or two-year lag — that is a coverage artefact, not a genuine data gap, and silently dropping those observations would bias the sample toward faster-reporting (typically wealthier) countries. `src/cleaning/build_master_dataset.py` implements this: for each country, it takes the most recent year with a valid consumption-expenditure observation, and flags which year that was.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src/visualisation')
master = pd.read_csv('../data/processed/master_consumer_spending.csv')
snapshot = pd.read_csv('../data/processed/latest_year_snapshot.csv')
print('Full panel:', master.shape)
print('Cross-sectional snapshot (one row per country):', snapshot.shape)
print('\nYear used per country in the snapshot (data lag, stated plainly):')
snapshot['Year'].value_counts().sort_index()

Full panel: (2387, 25)
Cross-sectional snapshot (one row per country): (182, 27)

Year used per country in the snapshot (data lag, stated plainly):


Year
2015      1
2016      1
2018      2
2020      1
2022      6
2023    171
Name: count, dtype: int64

## What Was NOT Silently Dropped

35 of 217 countries (16%) have **no** consumption-expenditure observation anywhere in 2013–2023 and are excluded from the snapshot and from every spending-based analysis in this project — not because their economies are uninteresting, but because World Bank has no reported figure for them. Notably this includes **Nigeria**, Africa's largest economy by GDP, whose national accounts reporting has well-documented gaps. This is stated here explicitly rather than smoothed over; the full exclusion list is in [`docs/DATA_COVERAGE.md`](../docs/DATA_COVERAGE.md).

In [5]:
has_consumption = master.groupby('ISO3')['household_consumption_expenditure_current_usd'].apply(lambda s: s.notna().any())
excluded = regions[regions['ISO3'].isin(has_consumption[~has_consumption].index)][['Country','ISO3','Project_007_Region']]
print(f'{len(excluded)} countries with zero consumption-expenditure data in 2013-2023:')
excluded.sort_values('Country').to_string(index=False)

35 countries with zero consumption-expenditure data in 2013-2023:


"                       Country ISO3     Project_007_Region\n                       Andorra  AND                 Europe\n           Antigua and Barbuda  ATG          Latin America\n                      Barbados  BRB          Latin America\n        British Virgin Islands  VGB          Latin America\n                Cayman Islands  CYM          Latin America\n               Channel Islands  CHI                 Europe\n                      Dominica  DMA          Latin America\n                       Eritrea  ERI                 Africa\n                     Gibraltar  GIB                 Europe\n                       Grenada  GRD          Latin America\n                        Guyana  GUY          Latin America\n                   Isle of Man  IMN                 Europe\n                       Jamaica  JAM          Latin America\n                        Jordan  JOR            Middle East\n     Korea, Dem. People's Rep.  PRK              East Asia\n                       Liberia  LBR    